In [1]:
import os
import io

import numpy as np

from typing import Tuple

import time
import cv2

from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torchvision
import torch.onnx
import torchsummary
print('torch.__version__', torch.__version__)

import onnx
import onnx2keras
import onnxruntime
from onnxsim import simplify
from onnx_tf.backend import prepare
print('onnx.__version__', onnx.__version__)

# import tvm
# import tvm.relay
# import tvm.contrib.graph_runtime as graph_runtime

from mobilenet_v2_tsm import MobileNetV2
from mobilenet_v2_keras_tsm import MobileNetV2TSM

import tensorflow as tf
print('tf.__version__', tf.__version__)
from tensorflow.python.keras import layers
from tensorflow.python.keras.engine import training
from keras.models import load_model
print('tf.keras.__version__', tf.keras.__version__)

import tensorflowjs as tfjs
print('tfjs.__version__', tfjs.__version__)

# import warnings
# warnings.filterwarnings('ignore')

torch.__version__ 1.4.0


/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/onnx-tensorflow/onnx_tf/common/__init__.py:96: UserWarning: onnx_tf.common.get_outputs_names is deprecated. It will be removed in future release. Use TensorflowGraph.get_outputs_names instead.
  warnings.warn(message)
Using TensorFlow backend.


onnx.__version__ 1.6.0
tf.__version__ 2.2.0
tf.keras.__version__ 2.3.0-tf
tfjs.__version__ 2.0.1


In [2]:
%load_ext autoreload
%autoreload 2

* torch2onnx:

In [3]:
SOFTMAX_THRES = 0
HISTORY_LOGIT = True
REFINE_OUTPUT = True

# def torch2tvm_module(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     torch_module.eval()
#     input_names = []
#     input_shapes = {}
#     with torch.no_grad():
#         for index, torch_input in enumerate(torch_inputs):
#             name = "i" + str(index)
#             input_names.append(name)
#             input_shapes[name] = torch_input.shape
#         buffer = io.BytesIO()
#         torch.onnx.export(torch_module, torch_inputs, buffer, input_names=input_names, output_names=["o" + str(i) for i in range(len(torch_inputs))])
#         outs = torch_module(*torch_inputs)
#         buffer.seek(0, 0)
#         onnx_model = onnx.load_model(buffer)
#         relay_module, params = tvm.relay.frontend.from_onnx(onnx_model, shape=input_shapes)
#     with tvm.relay.build_config(opt_level=3):
#         graph, tvm_module, params = tvm.relay.build(relay_module, target, params=params)
#     return graph, tvm_module, params


# def torch2executor(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     prefix = f"mobilenet_tsm_tvm_{target}"
#     lib_fname = f'{prefix}.tar'
#     graph_fname = f'{prefix}.json'
#     params_fname = f'{prefix}.params'
#     if os.path.exists(lib_fname) and os.path.exists(graph_fname) and os.path.exists(params_fname):
#         with open(graph_fname, 'rt') as f:
#             graph = f.read()
#         tvm_module = tvm.module.load(lib_fname)
#         params = tvm.relay.load_param_dict(bytearray(open(params_fname, 'rb').read()))
#     else:
#         graph, tvm_module, params = torch2tvm_module(torch_module, torch_inputs, target)
#         tvm_module.export_library(lib_fname)
#         with open(graph_fname, 'wt') as f:
#             f.write(graph)
#         with open(params_fname, 'wb') as f:
#             f.write(tvm.relay.save_param_dict(params))

#     ctx = tvm.gpu() if target.startswith('cuda') else tvm.cpu()
#     graph_module = graph_runtime.create(graph, tvm_module, ctx)
#     for pname, pvalue in params.items():
#         graph_module.set_input(pname, pvalue)

#     def executor(inputs: Tuple[tvm.nd.NDArray]):
#         for index, value in enumerate(inputs):
#             graph_module.set_input(index, value)
#         graph_module.run()
#         return tuple(graph_module.get_output(index) for index in range(len(inputs)))

#     return executor, ctx


# def get_executor(use_gpu=True):
#     torch_module = MobileNetV2(n_class=27)
#     if not os.path.exists("mobilenetv2_jester_online.pth.tar"):  # checkpoint not downloaded
#         print('Downloading PyTorch checkpoint...')
#         import urllib.request
#         url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
#         urllib.request.urlretrieve(url, './mobilenetv2_jester_online.pth.tar')
#     torch_module.load_state_dict(torch.load("mobilenetv2_jester_online.pth.tar"))
#     torch_inputs = (torch.rand(1, 3, 224, 224),
#                     torch.zeros([1, 3, 56, 56]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 20, 7, 7]),
#                     torch.zeros([1, 20, 7, 7]))
#     if use_gpu:
#         target = 'cuda'
#     else:
#         target = 'llvm -mcpu=cortex-a72 -target=armv7l-linux-gnueabihf'
#     return torch2executor(torch_module, torch_inputs, target)


def transform(frame: np.ndarray):
    # 480, 640, 3, 0 ~ 255
    frame = cv2.resize(frame, (224, 224))  # (224, 224, 3) 0 ~ 255
    frame = frame / 255.0  # (224, 224, 3) 0 ~ 1.0
    frame = np.transpose(frame, axes=[2, 0, 1])  # (3, 224, 224) 0 ~ 1.0
    frame = np.expand_dims(frame, axis=0)  # (1, 3, 480, 640) 0 ~ 1.0
    return frame


class GroupScale(object):
    """ Rescales the input PIL.Image to the given 'size'.
    'size' will be the size of the smaller edge.
    For example, if height > width, then image will be
    rescaled to (size * height / width, size)
    size: size of the smaller edge
    interpolation: Default: PIL.Image.BILINEAR
    """

    def __init__(self, size, interpolation=Image.BILINEAR):
        self.worker = torchvision.transforms.Scale(size, interpolation)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class GroupCenterCrop(object):
    def __init__(self, size):
        self.worker = torchvision.transforms.CenterCrop(size)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class Stack(object):

    def __init__(self, roll=False):
        self.roll = roll

    def __call__(self, img_group):
        if img_group[0].mode == 'L':
            return np.concatenate([np.expand_dims(x, 2) for x in img_group], axis=2)
        elif img_group[0].mode == 'RGB':
            if self.roll:
                return np.concatenate([np.array(x)[:, :, ::-1] for x in img_group], axis=2)
            else:
                return np.concatenate(img_group, axis=2)


class ToTorchFormatTensor(object):
    """ Converts a PIL.Image (RGB) or numpy.ndarray (H x W x C) in the range [0, 255]
    to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] """

    def __init__(self, div=True):
        self.div = div

    def __call__(self, pic):
        if isinstance(pic, np.ndarray):
            # handle numpy array
            img = torch.from_numpy(pic).permute(2, 0, 1).contiguous()
        else:
            # handle PIL Image
            img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
            img = img.view(pic.size[1], pic.size[0], len(pic.mode))
            # put it from HWC to CHW format
            # yikes, this transpose takes 80% of the loading time/CPU
            img = img.transpose(0, 1).transpose(0, 2).contiguous()
        return img.float().div(255) if self.div else img.float()


class GroupNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        rep_mean = self.mean * (tensor.size()[0] // len(self.mean))
        rep_std = self.std * (tensor.size()[0] // len(self.std))

        # TODO: make efficient
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)

        return tensor


def get_transform():
    cropping = torchvision.transforms.Compose([
        GroupScale(256),
        GroupCenterCrop(224),
    ])
    transform = torchvision.transforms.Compose([
        cropping,
        Stack(roll=False),
        ToTorchFormatTensor(div=True),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform

catigories = [
    "Doing other things",  # 0
    "Drumming Fingers",  # 1
    "No gesture",  # 2
    "Pulling Hand In",  # 3
    "Pulling Two Fingers In",  # 4
    "Pushing Hand Away",  # 5
    "Pushing Two Fingers Away",  # 6
    "Rolling Hand Backward",  # 7
    "Rolling Hand Forward",  # 8
    "Shaking Hand",  # 9
    "Sliding Two Fingers Down",  # 10
    "Sliding Two Fingers Left",  # 11
    "Sliding Two Fingers Right",  # 12
    "Sliding Two Fingers Up",  # 13
    "Stop Sign",  # 14
    "Swiping Down",  # 15
    "Swiping Left",  # 16
    "Swiping Right",  # 17
    "Swiping Up",  # 18
    "Thumb Down",  # 19
    "Thumb Up",  # 20
    "Turning Hand Clockwise",  # 21
    "Turning Hand Counterclockwise",  # 22
    "Zooming In With Full Hand",  # 23
    "Zooming In With Two Fingers",  # 24
    "Zooming Out With Full Hand",  # 25
    "Zooming Out With Two Fingers"  # 26
]


n_still_frame = 0

def process_output(idx_, history):
    # idx_: the output of current frame
    # history: a list containing the history of predictions
    if not REFINE_OUTPUT:
        return idx_, history

    max_hist_len = 20  # max history buffer

    # mask out illegal action
    if idx_ in [7, 8, 21, 22, 3]:
        idx_ = history[-1]

    # use only single no action class
    if idx_ == 0:
        idx_ = 2
    
    # history smoothing
    if idx_ != history[-1]:
        if not (history[-1] == history[-2]): #  and history[-2] == history[-3]):
            idx_ = history[-1]
    

    history.append(idx_)
    history = history[-max_hist_len:]

    return history[-1], history

In [4]:
def torch2onnx(
    torch_module: torch.nn.Module, 
    torch_inputs: Tuple[torch.Tensor, ...], 
    onnx_path
):
    torch_module.eval()
    input_names = []
    input_shapes = {}
    with torch.no_grad():
        for index, torch_input in enumerate(torch_inputs):
            name = "i" + str(index)
            input_names.append(name)
            input_shapes[name] = torch_input.shape
        with open(onnx_path, 'wb') as model_file:
            torch.onnx.export(
                torch_module, 
                torch_inputs, 
                model_file, 
                input_names=input_names, 
                output_names=["o" + str(i) for i in range(len(torch_inputs))],
                opset_version=10
            )

* Load the model:

In [5]:
os.makedirs('./models', exist_ok=True)
TORCH_MODEL_PATH= './models/mobilenetv2_jester_online.pth.tar'
torch_module = MobileNetV2(n_class=27)
if not os.path.exists(TORCH_MODEL_PATH):  # checkpoint not downloaded
    print('Downloading PyTorch checkpoint...')
    import urllib.request
    url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
    urllib.request.urlretrieve(url, TORCH_MODEL_PATH)
torch_module.load_state_dict(torch.load(TORCH_MODEL_PATH))
torch_module.eval()

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

* Convert to ONNX:

In [6]:
ONNX_MODEL_PATH = './models/jestnet.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32),
                torch.zeros([1, 3, 56, 56], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32))

for param in torch_module.parameters():
    param = param.float()

for module in torch_module.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=torch_module, 
    torch_inputs=torch_inputs, 
    onnx_path=ONNX_MODEL_PATH
)

/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/mobilenet_v2_tsm.py:95: TracerWarning: Converting a tensor to a Python index might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x1, x2 = x[:, : c // 8], x[:, c // 8:]


* Load from ONNX:

In [7]:
with open(ONNX_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)
onnx.checker.check_model(onnx_model)

In [8]:
print(onnx.helper.printable_graph(onnx_model.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
  %i1[FLOAT, 1x3x56x56]
  %i2[FLOAT, 1x4x28x28]
  %i3[FLOAT, 1x4x28x28]
  %i4[FLOAT, 1x8x14x14]
  %i5[FLOAT, 1x8x14x14]
  %i6[FLOAT, 1x8x14x14]
  %i7[FLOAT, 1x12x14x14]
  %i8[FLOAT, 1x12x14x14]
  %i9[FLOAT, 1x20x7x7]
  %i10[FLOAT, 1x20x7x7]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %features.0.0.weight[FLOAT, 32x3x3x3]
  %features.0.1.bias[FLOAT, 32]
  %features.0.1.num_batches_tracked[INT64, scalar]
  %features.0.1.running_mean[FLOAT, 32]
  %features.0.1.running_var[FLOAT, 32]
  %features.0.1.weight[FLOAT, 32]
  %features.1.conv.0.weight[FLOAT, 32x1x3x3]
  %features.1.conv.1.bias[FLOAT, 32]
  %features.1.conv.1.num_batches_tracked[FLOAT, scalar]
  %features.1.conv.1.running_mean[FLOAT, 32]
  %features.1.conv.1.running_var[FLOAT, 32]
  %features.1.conv.1.weight[FLOAT, 32]
  %features.1.conv.3.weight[FLOAT, 16x32x1x1]
  %features.1.conv.4.bias[FLOAT, 16]
  %features.1.conv.4.num_batches_tracke

* [optional] Simplify:

In [9]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [10]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9211341
After 8981232


In [11]:
print(onnx.helper.printable_graph(model_simp.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
  %i1[FLOAT, 1x3x56x56]
  %i2[FLOAT, 1x4x28x28]
  %i3[FLOAT, 1x4x28x28]
  %i4[FLOAT, 1x8x14x14]
  %i5[FLOAT, 1x8x14x14]
  %i6[FLOAT, 1x8x14x14]
  %i7[FLOAT, 1x12x14x14]
  %i8[FLOAT, 1x12x14x14]
  %i9[FLOAT, 1x20x7x7]
  %i10[FLOAT, 1x20x7x7]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %351[INT64, 1]
  %360[INT64, 1]
  %390[INT64, 1]
  %399[INT64, 1]
  %421[INT64, 1]
  %430[INT64, 1]
  %460[INT64, 1]
  %469[INT64, 1]
  %491[INT64, 1]
  %500[INT64, 1]
  %522[INT64, 1]
  %531[INT64, 1]
  %561[INT64, 1]
  %570[INT64, 1]
  %592[INT64, 1]
  %601[INT64, 1]
  %631[INT64, 1]
  %640[INT64, 1]
  %662[INT64, 1]
  %671[INT64, 1]
  %788[FLOAT, 32x3x3x3]
  %790[FLOAT, 32]
  %792[FLOAT, 32x1x3x3]
  %794[FLOAT, 32]
  %796[FLOAT, 16x32x1x1]
  %798[FLOAT, 16]
  %800[FLOAT, 96x16x1x1]
  %802[FLOAT, 96]
  %804[FLOAT, 96x1x3x3]
  %806[FLOAT, 96]
  %808[FLOAT, 24x96x1x1]
  %810[FLOAT, 24]
  %812[FLOAT, 144x24x1x1]
  %

In [12]:
ONNX_SIMPLE_MODEL_PATH = './models/jestnet_simple.onnx'
onnx.save(model_simp, ONNX_SIMPLE_MODEL_PATH)

* ONNX runtime check:

In [13]:
ort_session = onnxruntime.InferenceSession(ONNX_SIMPLE_MODEL_PATH)

In [14]:
input_names = [ort_session.get_inputs()[i].name for i in range(len(ort_session.get_inputs()))]
input_names

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']

In [15]:
output_names = [ort_session.get_outputs()[i].name for i in range(len(ort_session.get_inputs()))]
output_names

['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']

In [16]:
np_inputs = [
    np.random.rand(1, 3, 224, 224),
    np.random.rand(1, 3, 56, 56),
    np.random.rand(1, 4, 28, 28),
    np.random.rand(1, 4, 28, 28),
    np.random.rand(1, 8, 14, 14),
    np.random.rand(1, 8, 14, 14),
    np.random.rand(1, 8, 14, 14),
    np.random.rand(1, 12, 14, 14),
    np.random.rand(1, 12, 14, 14),
    np.random.rand(1, 20, 7, 7),
    np.random.rand(1, 20, 7, 7)
]
np_inputs = [np_input.astype(np.float32) for np_input in np_inputs]

In [17]:
%%time
outputs = ort_session.run(output_names, {input_names[i]: np_inputs[i] for i in range(len(np_inputs))})

CPU times: user 20.1 ms, sys: 14.3 ms, total: 34.4 ms
Wall time: 8.64 ms


In [18]:
for i in range(len(output_names)):
    print(outputs[i].shape)

(1, 27)
(1, 3, 56, 56)
(1, 4, 28, 28)
(1, 4, 28, 28)
(1, 8, 14, 14)
(1, 8, 14, 14)
(1, 8, 14, 14)
(1, 12, 14, 14)
(1, 12, 14, 14)
(1, 20, 7, 7)
(1, 20, 7, 7)


* tflite2onnx:

In [19]:
import tflite2onnx

tflite_path = '/Users/izakharkin/Desktop/inclusio/Inclusio/mediapipe/mediapipe/models/hand_landmark.tflite'
onnx_path = './models/hand_landmark.onnx'

tflite2onnx.convert(tflite_path, onnx_path)

NotImplementedError: Unsupported TFLite OP: 6

* ONNX -> TensorFlow:

In [19]:
tf_rep = prepare(onnx_model)

2020-07-05 21:59:21,665 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:21,666 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-07-05 21:59:21,666 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:21,667 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:21,667 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:21,669 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:21,670 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-07-05 21:59:21,671 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 2

Instructions for updating:
Create a `tf.sparse.SparseTensor` and use `tf.sparse.to_dense` instead.


In [20]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']
-----
['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, 'features.0.0.weight': <tf.Tensor 'features.0.0.weight:0' shape=(32, 3, 3, 3) dtype=float32>, 'features.0.1.bias': <tf.Tensor 'features.0.1.bias:0' shape=(32,) dtype=float32>, 'features.0.1.num_batches_tracked': <tf.Tensor 'features.0.1.num_batches_tracked:0' shape=() dtype=int64>, 'features.0.1.running_mean': <tf.Tensor 'features.0.1.running_mean:0' shape=(32,) dtype=float32>, 'features.0.1.running_var': <tf.Tensor 'features.0.1.running_var:0' shape=(32,) dtype=float32>, 'features.0.1.weight': <tf.Tensor 'features.0.1.weight:0' shape=(32,) dtype=float32>, 'features.1.conv.0.weight': <tf.Tensor 'features.1.conv.0.weight:0' shape=(32, 1, 3, 3) dtype=float32>, 'features.1.conv.1.

In [21]:
TF_MODEL_PATH = './models/jestnet_tf.pb'
tf_rep.export_graph(TF_MODEL_PATH)

* Simplified -> TensorFlow:

In [22]:
tf_rep = prepare(model_simp, strict=False)

2020-07-05 21:59:28,995 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:28,997 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-07-05 21:59:28,998 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:28,999 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:28,999 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:29,000 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:59:29,001 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-07-05 21:59:29,003 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 2

In [23]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']
-----
['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, '351': <tf.Tensor '351:0' shape=(1,) dtype=int64>, '360': <tf.Tensor '360:0' shape=(1,) dtype=int64>, '390': <tf.Tensor '390:0' shape=(1,) dtype=int64>, '399': <tf.Tensor '399:0' shape=(1,) dtype=int64>, '421': <tf.Tensor '421:0' shape=(1,) dtype=int64>, '430': <tf.Tensor '430:0' shape=(1,) dtype=int64>, '460': <tf.Tensor '460:0' shape=(1,) dtype=int64>, '469': <tf.Tensor '469:0' shape=(1,) dtype=int64>, '491': <tf.Tensor '491:0' shape=(1,) dtype=int64>, '500': <tf.Tensor '500:0' shape=(1,) dtype=int64>, '522': <tf.Tensor '522:0' shape=(1,) dtype=int64>, '531': <tf.Tensor '531:0' shape=(1,) dtype=int64>, '561': <tf.Tensor '561:0' shape=(1,) dtype=int64>, '570': <tf.Tensor '570:

In [24]:
TF_SIMPLE_MODEL_PATH = './models/jestnet_simple_tf.pb'
tf_rep.export_graph(TF_SIMPLE_MODEL_PATH)

* TensorFlow -> tf.js:

```
tensorflowjs_converter --input_format=tf_frozen_model --output_node_names='o0,o1,o2,o3,o4,o5,o6,o7,o8,o9,o10' ./jestnet_tf.pb ./jestnet
```

```
tensorflowjs_converter --input_format=tf_frozen_model --output_node_names='o0,o1,o2,o3,o4,o5,o6,o7,o8,o9,o10' ./jestnet_simple_tf.pb ./jestnet_simple
```

* TensorFlow model speed check:

In [25]:
tf.compat.v1.disable_eager_execution()

[Very useful link](https://blog.metaflow.fr/tensorflow-how-to-freeze-a-model-and-serve-it-with-a-python-api-d4f3596b3adc)

In [26]:
def load_graph(frozen_graph_filename):
    # We load the protobuf file from the disk and parse it to retrieve the 
    # unserialized graph_def
    with tf.io.gfile.GFile(frozen_graph_filename, "rb") as f:
        graph_def = tf.compat.v1.GraphDef()
        graph_def.ParseFromString(f.read())

    # Then, we import the graph_def into a new Graph and returns it 
    with tf.Graph().as_default() as graph:
        # The name var will prefix every op/nodes in your graph
        # Since we load everything in a new graph, this is not needed
        tf.import_graph_def(graph_def, name="prefix")
    return graph

In [27]:
TF_SIMPLE_MODEL_PATH = './models/jestnet_simple_tf.pb'
graph = load_graph(TF_SIMPLE_MODEL_PATH)

In [28]:
dir(graph)

['_ControlDependenciesController',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_add_control_dependencies',
 '_add_device_to_stack',
 '_add_function',
 '_add_new_tf_operations',
 '_add_op',
 '_apply_device_functions',
 '_as_graph_def',
 '_as_graph_element_locked',
 '_attr_scope',
 '_attr_scope_map',
 '_auto_cast_variable_read_dtype',
 '_bcast_grad_args_cache',
 '_building_function',
 '_c_graph',
 '_check_not_finalized',
 '_collections',
 '_colocate_with_for_gradient',
 '_colocation_stack',
 '_container',
 '_control_dependencies_for_inputs',
 '_control_dependencies_stack',
 '_control_flow_context',
 '_copy_functions_to_graph_def',
 '_create_op_from_tf_operation',
 '

In [29]:
# graph.get_operations()

In [30]:
# i0 = tf.compat.v1.placeholder(tf.float32, shape=[None, 3, 224, 224], name="i0")
# i1 = tf.compat.v1.placeholder(tf.float32, shape=[None, 3, 56, 56], name="i1")
# i2 = tf.compat.v1.placeholder(tf.float32, shape=[None, 4, 28, 28], name="i2")
# i3 = tf.compat.v1.placeholder(tf.float32, shape=[None, 4, 28, 28], name="i3")
# i4 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i4")
# i5 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i5")
# i6 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i6")
# i7 = tf.compat.v1.placeholder(tf.float32, shape=[None, 12, 14, 14], name="i7")
# i8 = tf.compat.v1.placeholder(tf.float32, shape=[None, 12, 14, 14], name="i8")
# i9 = tf.compat.v1.placeholder(tf.float32, shape=[None, 20, 7, 7], name="i9")
# i10 = tf.compat.v1.placeholder(tf.float32, shape=[None, 20, 7, 7], name="i10")

tf_buffer = [
    np.zeros([1, 3, 224, 224]),
    np.zeros([1, 3, 56, 56]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 20, 7, 7]),
    np.zeros([1, 20, 7, 7])
]

# We can verify that we can access the list of operations in the graph
# for op in graph.get_operations():
#     if ':' in op.name:
#         print(op.name)
#     if '/i' in op.name:
#         print(op.name)
#     if '/o' in op.name:
#         print(op.name)
#     # prefix/Placeholder/inputs_placeholder
#     # ...
#     # prefix/Accuracy/predictions

# We access the input and output nodes
tf_inputs = []
tf_outputs = []
for i in range(len(tf_buffer)):
    tf_inputs.append(graph.get_tensor_by_name(f'prefix/i{i}:0'))
    tf_outputs.append(graph.get_tensor_by_name(f'prefix/o{i}:0'))

In [31]:
with tf.compat.v1.Session(graph=graph) as sess:
    # Note: we don't nee to initialize/restore anything
    # There is no Variables in this graph, only hardcoded constants 
    output = sess.run(tf_outputs, feed_dict={
        tf_inputs[i]: tf_buffer[i] for i in range(len(tf_buffer))
    })
    print(len(output))

11


In [32]:
with tf.compat.v1.Session(graph=graph) as sess:
    for _ in range(100):
        begin = time.time()
        output = sess.run(tf_outputs, feed_dict={
            tf_inputs[i]: tf_buffer[i] for i in range(len(tf_buffer))
        })
        print('Curr time (s):', time.time() - begin)

Curr time (s): 11.18132209777832
Curr time (s): 0.09156489372253418
Curr time (s): 0.09984803199768066
Curr time (s): 0.12664484977722168
Curr time (s): 0.10034584999084473
Curr time (s): 0.09255194664001465
Curr time (s): 0.09465384483337402
Curr time (s): 0.09874415397644043
Curr time (s): 0.09171104431152344
Curr time (s): 0.09575605392456055
Curr time (s): 0.0969088077545166
Curr time (s): 0.09137105941772461
Curr time (s): 0.09602117538452148
Curr time (s): 0.09480595588684082
Curr time (s): 0.09145927429199219
Curr time (s): 0.09600830078125
Curr time (s): 0.09587383270263672
Curr time (s): 0.09068512916564941
Curr time (s): 0.0942690372467041
Curr time (s): 0.09707307815551758
Curr time (s): 0.0928652286529541
Curr time (s): 0.09902524948120117
Curr time (s): 0.10506129264831543
Curr time (s): 0.09199690818786621
Curr time (s): 0.09520721435546875
Curr time (s): 0.0942239761352539
Curr time (s): 0.10647320747375488
Curr time (s): 0.10465288162231445
Curr time (s): 0.103567123413

* `torchvision.models.mobilenet_v2` sanity check:

In [4]:
mobilenetv2_torch = torchvision.models.mobilenet_v2(pretrained=False, num_classes=27)
mobilenetv2_torch

MobileNetV2(
  (features): Sequential(
    (0): ConvBNReLU(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=Tr

In [6]:
torch_module

In [36]:
MV2_MODEL_PATH = './models/mobilenet_v2.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32))

for param in mobilenetv2_torch.parameters():
    param = param.float()

for module in mobilenetv2_torch.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=mobilenetv2_torch, 
    torch_inputs=torch_inputs, 
    onnx_path=MV2_MODEL_PATH
)

In [37]:
with open(MV2_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)
onnx.checker.check_model(onnx_model)

In [38]:
# print(onnx.helper.printable_graph(onnx_model.graph))

In [39]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [40]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9203331
After 8977491


In [41]:
# print(onnx.helper.printable_graph(model_simp.graph))

In [42]:
MV2_SIMPLE_MODEL_PATH = './models/mv2_simple.onnx'
onnx.save(model_simp, MV2_SIMPLE_MODEL_PATH)

In [43]:
ort_session = onnxruntime.InferenceSession(MV2_SIMPLE_MODEL_PATH)
ort_session

In [44]:
tf_rep = prepare(model_simp)
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

2020-06-28 23:38:30,849 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,850 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-06-28 23:38:30,851 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,852 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,853 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,854 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,855 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-06-28 23:38:30,857 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 2

['i0']
-----
['o0']
-----
{'classifier.1.bias': <tf.Tensor 'classifier.1.bias:0' shape=(27,) dtype=float32>, 'classifier.1.weight': <tf.Tensor 'classifier.1.weight:0' shape=(27, 1280) dtype=float32>, '467': <tf.Tensor '467:0' shape=(32, 3, 3, 3) dtype=float32>, '469': <tf.Tensor '469:0' shape=(32,) dtype=float32>, '471': <tf.Tensor '471:0' shape=(32, 1, 3, 3) dtype=float32>, '473': <tf.Tensor '473:0' shape=(32,) dtype=float32>, '475': <tf.Tensor '475:0' shape=(16, 32, 1, 1) dtype=float32>, '477': <tf.Tensor '477:0' shape=(16,) dtype=float32>, '479': <tf.Tensor '479:0' shape=(96, 16, 1, 1) dtype=float32>, '481': <tf.Tensor '481:0' shape=(96,) dtype=float32>, '483': <tf.Tensor '483:0' shape=(96, 1, 3, 3) dtype=float32>, '485': <tf.Tensor '485:0' shape=(96,) dtype=float32>, '487': <tf.Tensor '487:0' shape=(24, 96, 1, 1) dtype=float32>, '489': <tf.Tensor '489:0' shape=(24,) dtype=float32>, '491': <tf.Tensor '491:0' shape=(144, 24, 1, 1) dtype=float32>, '493': <tf.Tensor '493:0' shape=(144,

In [45]:
TF_SIMPLE_MODEL_PATH = './models/mv2_simple_tf.pb'
tf_rep.export_graph(TF_SIMPLE_MODEL_PATH)

* Keras -> tf.js MobileNetV2 sanity check:

In [21]:
from tensorflow.python.keras import backend
backend.set_image_data_format('channels_last')

keras_mv2 = tf.keras.applications.MobileNetV2(
    input_shape=None,
    alpha=1.0,
    include_top=True,
    weights=None,
    input_tensor=None,
    pooling=None,
    classes=27
)

In [22]:
# keras_mv2.compile()
keras_mv2.summary()

Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
Conv1_pad (ZeroPadding2D)       (None, 225, 225, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 112, 112, 32) 864         Conv1_pad[0][0]                  
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 112, 112, 32) 128         Conv1[0][0]                      
_______________________________________________________________________________

In [23]:
keras_mv2.save('./models/keras_mv2.h5')  # creates a HDF5 file 'my_model.h5'
del keras_mv2  # deletes the existing model

# returns a compiled model
# identical to the previous one
keras_mv2 = load_model('./models/keras_mv2.h5')
keras_mv2.summary()

Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
Conv1_pad (ZeroPadding2D)       (None, 225, 225, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 112, 112, 32) 864         Conv1_pad[0][0]                  
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 112, 112, 32) 128         Conv1[0][0]                      
_______________________________________________________________________________

In [24]:
# for _ in range(30):
#     begin = time.time()
#     y_pred = keras_mv2(np.zeros([1, 224, 224, 3]))
#     print(time.time() - begin)

In [25]:
### NOTE: DIDN'T WORK!!! Use command line tool
# tfjs.converters.save_keras_model(keras_mv2, './models/keras_mv2/')

`tensorflowjs_converter --input_format keras ./keras_mv2.h5 ./keras_mobilenetv2/`

* Keras model with multiple inputs/outputs and slices:

[keras multiple inputs / outputs](https://github.com/tensorflow/tensorflow/issues/34114)

In [35]:
# kvar = tf.keras.backend.zeros((1,10), name='i0')
# ekvar = tf.keras.backend.eval(kvar)

In [38]:
keras_buffer = [
    np.zeros([1, 3, 224, 224]),
    np.zeros([1, 3, 56, 56]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 20, 7, 7]),
    np.zeros([1, 20, 7, 7])
]

inputs = [
    layers.Input(keras_buffer[i].shape[1:], name=f'i{i}') 
    for i in range(len(keras_buffer))
]

outputs = [
    tf.keras.layers.Add()([
        tf.slice(inputs[i], [0,0,0,0], [-1,-1,5,5]), 
        tf.slice(inputs[i], [0,0,0,0], [-1,-1,5,5])
    ]) 
    for i in range(len(keras_buffer))
]

model = training.Model(inputs, outputs, name='model_1')

x = {f'i{i}': keras_buffer[i] for i in range(len(keras_buffer))}

y_pred = model(x)

In [39]:
print(len(y_pred))
print([y_pred[i].shape for i in range(len(y_pred))])

11
[TensorShape([1, 3, 5, 5]), TensorShape([1, 3, 5, 5]), TensorShape([1, 4, 5, 5]), TensorShape([1, 4, 5, 5]), TensorShape([1, 8, 5, 5]), TensorShape([1, 8, 5, 5]), TensorShape([1, 8, 5, 5]), TensorShape([1, 12, 5, 5]), TensorShape([1, 12, 5, 5]), TensorShape([1, 20, 5, 5]), TensorShape([1, 20, 5, 5])]


In [40]:
model.compile()
model.summary()

ValueError: The model cannot be compiled because it has no loss to optimize.

In [ ]:
model.save('./models/add_model.h5')
del model
model = load_model('./models/add_model.h5')
model.summary()

In [12]:
tfjs.converters.save_keras_model(model, './models/add_model_2/')

/Users/izakharkin/opt/anaconda3/envs/tf20/lib/python3.7/site-packages/tensorflowjs/converters/keras_h5_conversion.py:122: H5pyDeprecationWarning: The default file mode will change to 'r' (read-only) in h5py 3.0. To suppress this warning, pass the mode you need to h5py.File(), or set the global default h5.get_config().default_file_mode, or set the environment variable H5PY_DEFAULT_READONLY=1. Available modes are: 'r', 'r+', 'w', 'w-'/'x', 'a'. See the docs for details.
  return h5py.File(h5file)


> `tensorflowjs_converter --input_format keras ./add_model.h5 ./add_model`

In [ ]:
{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice_10", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice_10", 
            "op": "Slice", 
            "input": ["i5", "Slice_10/begin", "Slice_10/size"], 
            "attr": {"Index": {"type": "DT_INT32"}, "T": {"type": "DT_FLOAT"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}}, 
    "name": "tf_op_layer_Slice_10", 
    "inbound_nodes": [[["i5", 0, 0, {}]]]
}


In [ ]:
{
    "class_name": "InputLayer", 
    "config": {
        "batch_input_shape": [null, 3, 224, 224], 
        "dtype": "float32", 
        "sparse": false, 
        "ragged": false, 
        "name": "i0"
    }, 
    "name": "i0", 
    "inbound_nodes": []
}

{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice", 
            "op": "Slice", 
            "input": ["i0", "Slice/begin", "Slice/size"], 
            "attr": {"T": {"type": "DT_FLOAT"}, "Index": {"type": "DT_INT32"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}
    }, 
    "name": "tf_op_layer_Slice", 
    "inbound_nodes": [[["i0", 0, 0, {}]]]
},
{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice_1", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice_1", 
            "op": "Slice", 
            "input": ["i0", "Slice_1/begin", "Slice_1/size"], 
            "attr": {"T": {"type": "DT_FLOAT"}, "Index": {"type": "DT_INT32"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}}, 
    "name": "tf_op_layer_Slice_1", 
    "inbound_nodes": [[["i0", 0, 0, {}]]]
}, 
{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice_2", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice_2", 
            "op": "Slice", 
            "input": ["i1", "Slice_2/begin", "Slice_2/size"], 
            "attr": {"T": {"type": "DT_FLOAT"}, "Index": {"type": "DT_INT32"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}}, 
    "name": "tf_op_layer_Slice_2", 
    "inbound_nodes": [[["i1", 0, 0, {}]]]
}


### Keras `MobileNetV2TSM`:

In [20]:
from tensorflow.python.keras import backend
backend.set_image_data_format('channels_last')

In [21]:
keras_buffer = [
    np.zeros([1, 224, 224, 3]),
    np.zeros([1, 56, 56, 3]),
    np.zeros([1, 28, 28, 4]),
    np.zeros([1, 28, 28, 4]),
    np.zeros([1, 14, 14, 8]),
    np.zeros([1, 14, 14, 8]),
    np.zeros([1, 14, 14, 8]),
    np.zeros([1, 14, 14, 12]),
    np.zeros([1, 14, 14, 12]),
    np.zeros([1, 7, 7, 20]),
    np.zeros([1, 7, 7, 20])
]

input_shapes = [tensor.shape[1:] for tensor in keras_buffer]
keras_tsm = MobileNetV2TSM(
    input_shapes=[tensor.shape[1:] for tensor in keras_buffer],
    alpha=1.0,
    weights=None,
    input_tensor=None,
    classes=27
)

test_input = {f'i{i}': keras_buffer[i] for i in range(len(keras_buffer))}

y_pred = keras_tsm(test_input)
print(len(y_pred))
print([y_pred[i].shape for i in range(len(y_pred))])

[(224, 224, 3), (56, 56, 3), (28, 28, 4), (28, 28, 4), (14, 14, 8), (14, 14, 8), (14, 14, 8), (14, 14, 12), (14, 14, 12), (7, 7, 20), (7, 7, 20)]
11
[TensorShape([1, 27]), TensorShape([1, 56, 56, 3]), TensorShape([1, 28, 28, 4]), TensorShape([1, 28, 28, 4]), TensorShape([1, 14, 14, 8]), TensorShape([1, 14, 14, 8]), TensorShape([1, 14, 14, 8]), TensorShape([1, 14, 14, 12]), TensorShape([1, 14, 14, 12]), TensorShape([1, 7, 7, 20]), TensorShape([1, 7, 7, 20])]


In [22]:
keras_tsm.compile()
keras_tsm.summary()

Model: "mobilenetv2_1.00_None"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
i0 (InputLayer)                 [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 111, 111, 32) 864         i0[0][0]                         
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 111, 111, 32) 128         Conv1[0][0]                      
__________________________________________________________________________________________________
Conv1_relu (ReLU)               (None, 111, 111, 32) 0           bn_Conv1[0][0]                   
______________________________________________________________________________

In [23]:
torchsummary.summary(torch_module, [tensor.shape[1:][::-1] for tensor in keras_buffer])

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
             ReLU6-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
             ReLU6-6         [-1, 32, 112, 112]               0
            Conv2d-7         [-1, 16, 112, 112]             512
       BatchNorm2d-8         [-1, 16, 112, 112]              32
  InvertedResidual-9         [-1, 16, 112, 112]               0
           Conv2d-10         [-1, 96, 112, 112]           1,536
      BatchNorm2d-11         [-1, 96, 112, 112]             192
            ReLU6-12         [-1, 96, 112, 112]               0
           Conv2d-13           [-1, 96, 56, 56]             864
      BatchNorm2d-14           [-1, 96,

In [24]:
# for _ in range(30):
#     begin = time.time()
#     y_pred = keras_tsm(test_input)
#     print(time.time() - begin)

In [25]:
keras_tsm.save('./models/keras_tsm.h5')
del keras_tsm

keras_tsm = load_model('./models/keras_tsm.h5')
keras_tsm.summary()

Model: "mobilenetv2_1.00_None"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
i0 (InputLayer)                 [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 111, 111, 32) 864         i0[0][0]                         
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 111, 111, 32) 128         Conv1[0][0]                      
__________________________________________________________________________________________________
Conv1_relu (ReLU)               (None, 111, 111, 32) 0           bn_Conv1[0][0]                   
______________________________________________________________________________

`tensorflowjs_converter --input_format keras ./keras_tsm.h5 ./keras_tsm/`

`tensorflowjs_converter --input_format tfjs_layers_model --output_format tfjs_graph_model ./keras_tsm/model.json ./keras_tsm_graph/`

* Correct slice model:

In [5]:
keras_buffer = [
    np.zeros([1, 3, 224, 224])
]

inputs = [
    layers.Input(keras_buffer[0].shape[1:], name=f'i{i}') 
    for i in range(len(keras_buffer))
]

# tf.keras.layers.Lambda(lambda x: x[:,:,:8,:8])(inputs[i])  # !!! Lambda didn't work woth tf.js converted
outputs = [
    tf.slice(inputs[i], [0,0,0,0], [-1,-1,8,8])
    for i in range(len(keras_buffer))
]

model = training.Model(inputs, outputs, name='model_1')


x = {f'i{i}': keras_buffer[i] for i in range(len(keras_buffer))}

y_pred = model(x)
print(len(y_pred))
print([y_pred[i].shape for i in range(len(y_pred))])

model.compile()
model.summary()

tfjs.converters.save_keras_model(model, './models/slice_model/')

AttributeError: module 'tensorflow.keras.backend' has no attribute 'slice'

* `from tensorflow.python import keras`
* `keras.utils.generic_utils.slice_arrays(input_a, start=0)`

* https://github.com/tensorflow/tensorflow/blob/e19e2d29d562724ead9e60e1ba4c4ffd91a0eb7a/tensorflow/python/keras/utils/generic_utils_test.py
    
* `from tensorflow.python.ops import array_ops`
* `array_ops.slice()  # same as tf.slice()`

* https://github.com/keras-team/keras/issues/890

* PyTorch -> Keras:

In [42]:
def pth2keras(pth_model, keras_model):
    m = {} # m : {'classifier.1.bias': np.array(...), ...}

    for k, v in pth_model.named_parameters():
        m[k] = v
    for k, v in pth_model.named_buffers(): # for batchnormalization
        m[k] = v
        
    print('torch model names:\n', m.keys())
    print('keras model names:')
    for layer in keras_model.layers:
        print(layer.name)

    with torch.no_grad():
        for layer in keras_model.layers:
            if isinstance(layer, DepthwiseConv2D):
                print(layer.name)
                weights = []
                weights.append(m[layer.name+'.weight'].permute(2, 3, 0, 1).data.numpy()) # weight
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].data.numpy()) # bias
                layer.set_weights(weights)
            elif isinstance(layer, Conv2D):
                # https://github.com/keras-team/keras/issues/8144
                # pth: (out_ch, in_ch, h, w)
                # tf/keras: (h, w, in_ch, out_ch)
                print(layer.name)
                weights = []
#                 print(m[layer.name+'.weight'].shape)
                weights.append(m[layer.name+'.weight'].permute(2, 3, 1, 0).data.numpy()) # weight
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].data.numpy()) # bias
                layer.set_weights(weights)
            elif isinstance(layer, BatchNormalization):
                print(layer.name)
                weights = []
                if layer.scale:
                    weights.append(m[layer.name+'.weight'].data.numpy()) # gamma
                if layer.center:
                    weights.append(m[layer.name+'.bias'].data.numpy()) # beta
                weights.append(m[layer.name+'.running_mean'].data.numpy()) # running_mean
                weights.append(m[layer.name+'.running_var'].data.numpy()) # running_var
                layer.set_weights(weights)
            elif isinstance(layer, Dense):
                print(layer.name)
                weights = []
                weights.append(m[layer.name+'.weight'].t().data.numpy())
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].data.numpy())
                layer.set_weights(weights)

In [44]:
# from tensorflow.keras.layers import (
#     Conv2D, Dense, BatchNormalization, 
#     DepthwiseConv2D, Input, 
# )
# from tensorflow.keras.models import Model


# class PthModel(nn.Sequential):
#     def __init__(self):
#         super(PthModel, self).__init__(
#             nn.Conv2d(3, 1, kernel_size=3, stride=1, padding=1, bias=False)
#         )

# pth_model = PthModel()
# inputs = Input(shape=[224, 224, 3])
# outputs = Conv2D(
#     data_format='channels_last', 
#     filters=1, 
#     kernel_size=3, 
#     strides=1, 
#     padding='same', 
#     use_bias=False, 
#     name='0'
# )(inputs)
# keras_model = Model(inputs=inputs, outputs=outputs)

In [45]:
# keras_model.get_weights()[0].shape

In [46]:
# pth2keras(pth_model=pth_model, keras_model=keras_model)

In [47]:
# pth_inputs = torch.randn([1, 3, 224, 224], dtype=torch.float32)
# keras_inputs = pth_inputs.permute(0, 2, 3, 1).numpy()

# pth_outputs = pth_model(pth_inputs)
# pth_outputs = pth_outputs.permute(0, 2, 3, 1).data.numpy() # keras output 과 형태가 맞도록 변형
# keras_outputs = keras_model.predict(keras_inputs)
# print(np.abs(pth_outputs-keras_outputs).max())

MobileNetV2 check:

In [52]:
mobilenetv2_torch = torchvision.models.mobilenet_v2(pretrained=False, width_mult=1.4, num_classes=27)
# torchsummary.summary(mobilenetv2_torch, input_size=(3, 224, 224))

In [53]:
# mobilenetv2_keras = tf.keras.applications.MobileNetV2(
#     input_shape=None,
#     input_tensor=None,
#     weights=None,
#     alpha=1.4,
#     include_top=True,
#     pooling=None,
#     classes=27
# )
# mobilenetv2_keras.summary()

In [54]:
# pth2keras(pth_model=mobilenetv2_torch, keras_model=mobilenetv2_keras)

In [55]:
import pytorch2keras

In [58]:
torch_inputs = [torch.rand(1, 3, 224, 224, dtype=torch.float32),]

k_model = pytorch2keras.pytorch_to_keras(
    mobilenetv2_torch, 
    args=torch_inputs[0],
    verbose=True
)

INFO:pytorch2keras:Converter is called.
DEBUG:pytorch2keras:Input_names:
DEBUG:pytorch2keras:['input_0']
DEBUG:pytorch2keras:Output_names:
DEBUG:pytorch2keras:['output_0']
INFO:onnx2keras:Converter is called.
DEBUG:onnx2keras:List input shapes:
DEBUG:onnx2keras:None
DEBUG:onnx2keras:List inputs:
DEBUG:onnx2keras:Input 0 -> input_0.
DEBUG:onnx2keras:List outputs:
DEBUG:onnx2keras:Output 0 -> output_0.
DEBUG:onnx2keras:Gathering weights to dictionary.
DEBUG:onnx2keras:Found weight classifier.1.bias with shape (27,).
DEBUG:onnx2keras:Found weight classifier.1.weight with shape (27, 1792).
DEBUG:onnx2keras:Found weight features.0.0.weight with shape (48, 3, 3, 3).
DEBUG:onnx2keras:Found weight features.0.1.bias with shape (48,).
DEBUG:onnx2keras:Found weight features.0.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.0.1.running_mean with shape (48,).
DEBUG:onnx2keras:Found weight features.0.1.running_var with shape (48,).
DEBUG:onnx2keras:Found weight features.0

DEBUG:onnx2keras:Found weight features.14.conv.0.1.weight with shape (816,).
DEBUG:onnx2keras:Found weight features.14.conv.1.0.weight with shape (816, 1, 3, 3).
DEBUG:onnx2keras:Found weight features.14.conv.1.1.bias with shape (816,).
DEBUG:onnx2keras:Found weight features.14.conv.1.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.14.conv.1.1.running_mean with shape (816,).
DEBUG:onnx2keras:Found weight features.14.conv.1.1.running_var with shape (816,).
DEBUG:onnx2keras:Found weight features.14.conv.1.1.weight with shape (816,).
DEBUG:onnx2keras:Found weight features.14.conv.2.weight with shape (224, 816, 1, 1).
DEBUG:onnx2keras:Found weight features.14.conv.3.bias with shape (224,).
DEBUG:onnx2keras:Found weight features.14.conv.3.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.14.conv.3.running_mean with shape (224,).
DEBUG:onnx2keras:Found weight features.14.conv.3.running_var with shape (224,).
DEBUG:onnx2keras:Found weight fe

DEBUG:onnx2keras:Found weight features.3.conv.1.1.weight with shape (192,).
DEBUG:onnx2keras:Found weight features.3.conv.2.weight with shape (32, 192, 1, 1).
DEBUG:onnx2keras:Found weight features.3.conv.3.bias with shape (32,).
DEBUG:onnx2keras:Found weight features.3.conv.3.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.3.conv.3.running_mean with shape (32,).
DEBUG:onnx2keras:Found weight features.3.conv.3.running_var with shape (32,).
DEBUG:onnx2keras:Found weight features.3.conv.3.weight with shape (32,).
DEBUG:onnx2keras:Found weight features.4.conv.0.0.weight with shape (192, 32, 1, 1).
DEBUG:onnx2keras:Found weight features.4.conv.0.1.bias with shape (192,).
DEBUG:onnx2keras:Found weight features.4.conv.0.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.4.conv.0.1.running_mean with shape (192,).
DEBUG:onnx2keras:Found weight features.4.conv.0.1.running_var with shape (192,).
DEBUG:onnx2keras:Found weight features.4.conv.0.1.

graph(%input_0 : Float(1, 3, 224, 224),
      %features.0.0.weight : Float(48, 3, 3, 3),
      %features.0.1.weight : Float(48),
      %features.0.1.bias : Float(48),
      %features.0.1.running_mean : Float(48),
      %features.0.1.running_var : Float(48),
      %features.0.1.num_batches_tracked : Long(),
      %features.1.conv.0.0.weight : Float(48, 1, 3, 3),
      %features.1.conv.0.1.weight : Float(48),
      %features.1.conv.0.1.bias : Float(48),
      %features.1.conv.0.1.running_mean : Float(48),
      %features.1.conv.0.1.running_var : Float(48),
      %features.1.conv.0.1.num_batches_tracked : Long(),
      %features.1.conv.1.weight : Float(24, 48, 1, 1),
      %features.1.conv.2.weight : Float(24),
      %features.1.conv.2.bias : Float(24),
      %features.1.conv.2.running_mean : Float(24),
      %features.1.conv.2.running_var : Float(24),
      %features.1.conv.2.num_batches_tracked : Long(),
      %features.2.conv.0.0.weight : Float(144, 24, 1, 1),
      %features.2.conv.0.

DEBUG:onnx2keras:Found weight features.4.conv.1.1.running_mean with shape (192,).
DEBUG:onnx2keras:Found weight features.4.conv.1.1.running_var with shape (192,).
DEBUG:onnx2keras:Found weight features.4.conv.1.1.weight with shape (192,).
DEBUG:onnx2keras:Found weight features.4.conv.2.weight with shape (48, 192, 1, 1).
DEBUG:onnx2keras:Found weight features.4.conv.3.bias with shape (48,).
DEBUG:onnx2keras:Found weight features.4.conv.3.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.4.conv.3.running_mean with shape (48,).
DEBUG:onnx2keras:Found weight features.4.conv.3.running_var with shape (48,).
DEBUG:onnx2keras:Found weight features.4.conv.3.weight with shape (48,).
DEBUG:onnx2keras:Found weight features.5.conv.0.0.weight with shape (288, 48, 1, 1).
DEBUG:onnx2keras:Found weight features.5.conv.0.1.bias with shape (288,).
DEBUG:onnx2keras:Found weight features.5.conv.0.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.5.conv.0.1.

...

DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: BatchNormalization
DEBUG:onnx2keras:node_name: 428
DEBUG:onnx2keras:node_params: {'epsilon': 9.999999747378752e-06, 'momentum': 0.8999999761581421, 'change_ordering': False, 'name_policy': None}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 427).
DEBUG:onnx2keras:Check input 1 (name features.14.conv.0.1.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.14.conv.0.1.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.14.conv.0.1.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBU

DEBUG:onnx2keras:Check input 0 (name 435).
DEBUG:onnx2keras:Check input 1 (name features.15.conv.0.1.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.15.conv.0.1.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.15.conv.0.1.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 4 (name features.15.conv.0.1.running_var).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:Output TF Layer -> Tensor("436/cond/Identity:0", shape=(None, 1344, 7, 7), dtype=float32)
DEBUG:onnx2keras:######
DEBUG:onnx2keras

DEBUG:onnx2keras:node_params: {'epsilon': 9.999999747378752e-06, 'momentum': 0.8999999761581421, 'change_ordering': False, 'name_policy': None}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 444).
DEBUG:onnx2keras:Check input 1 (name features.16.conv.0.1.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.16.conv.0.1.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.16.conv.0.1.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 4 (name features.16.conv.0.1.running_var).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weigh

DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: BatchNormalization
DEBUG:onnx2keras:node_name: 454
DEBUG:onnx2keras:node_params: {'epsilon': 9.999999747378752e-06, 'momentum': 0.8999999761581421, 'change_ordering': False, 'name_policy': None}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 453).
DEBUG:onnx2keras:Check input 1 (name features.17.conv.0.1.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.17.conv.0.1.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.17.conv.0.1.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBU

DEBUG:onnx2keras:Check input 0 (name 461).
DEBUG:onnx2keras:Check input 1 (name features.18.1.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.18.1.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.18.1.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 4 (name features.18.1.running_var).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:Output TF Layer -> Tensor("462/cond/Identity:0", shape=(None, 1792, 7, 7), dtype=float32)
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Conver

In [59]:
k_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_0 (InputLayer)            [(None, 3, 224, 224) 0                                            
__________________________________________________________________________________________________
315_pad (ZeroPadding2D)         (None, 3, 226, 226)  0           input_0[0][0]                    
__________________________________________________________________________________________________
315 (Conv2D)                    (None, 48, 112, 112) 1296        315_pad[0][0]                    
__________________________________________________________________________________________________
316 (BatchNormalization)        (None, 48, 112, 112) 192         315[0][0]                        
____________________________________________________________________________________________

* tsm:

In [65]:
torch_inputs = [torch.rand(1, 3, 224, 224, dtype=torch.float32),
                torch.zeros([1, 3, 56, 56], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32)]

k_model = pytorch2keras.pytorch_to_keras(
    torch_module, 
    args=torch_inputs,
    verbose=True,
    do_constant_folding=True
)

INFO:pytorch2keras:Converter is called.
DEBUG:pytorch2keras:Input_names:
DEBUG:pytorch2keras:['input_0', 'input_1', 'input_2', 'input_3', 'input_4', 'input_5', 'input_6', 'input_7', 'input_8', 'input_9', 'input_10']
DEBUG:pytorch2keras:Output_names:
DEBUG:pytorch2keras:['output_0', 'output_1', 'output_2', 'output_3', 'output_4', 'output_5', 'output_6', 'output_7', 'output_8', 'output_9', 'output_10']


RuntimeError: Unsupported: ONNX export of Slice with dynamic inputs. DynamicSlice is a deprecated experimental op. Please use statically allocated variables or export to a higher opset version.

*NOTE: `/opt/anaconda3/envs/tf20/lib/python3.7/site-packages/onnx2keras/reshape_layers.py` was changed to fix the bugs!*
[onnx2keras](https://github.com/nerox8664/onnx2keras)

In [1]:
import onnx2keras
import torch
import onnx

torch_inputs = [torch.rand(1, 3, 224, 224, dtype=torch.float32),
                torch.zeros([1, 3, 56, 56], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32)]

onnx_path = './models/jestnet.onnx'
onnx_model = onnx.load(onnx_path)
k_model = onnx2keras.onnx_to_keras(
    onnx_model, 
    input_names=[f'i{i}' for i in range(len(torch_inputs))],
    name_policy='short'
)

INFO:onnx2keras:Converter is called.
DEBUG:onnx2keras:List input shapes:
DEBUG:onnx2keras:None
DEBUG:onnx2keras:List inputs:
DEBUG:onnx2keras:Input 0 -> i0.
DEBUG:onnx2keras:Input 1 -> i1.
DEBUG:onnx2keras:Input 2 -> i2.
DEBUG:onnx2keras:Input 3 -> i3.
DEBUG:onnx2keras:Input 4 -> i4.
DEBUG:onnx2keras:Input 5 -> i5.
DEBUG:onnx2keras:Input 6 -> i6.
DEBUG:onnx2keras:Input 7 -> i7.
DEBUG:onnx2keras:Input 8 -> i8.
DEBUG:onnx2keras:Input 9 -> i9.
DEBUG:onnx2keras:Input 10 -> i10.
DEBUG:onnx2keras:List outputs:
DEBUG:onnx2keras:Output 0 -> o0.
DEBUG:onnx2keras:Output 1 -> o1.
DEBUG:onnx2keras:Output 2 -> o2.
DEBUG:onnx2keras:Output 3 -> o3.
DEBUG:onnx2keras:Output 4 -> o4.
DEBUG:onnx2keras:Output 5 -> o5.
DEBUG:onnx2keras:Output 6 -> o6.
DEBUG:onnx2keras:Output 7 -> o7.
DEBUG:onnx2keras:Output 8 -> o8.
DEBUG:onnx2keras:Output 9 -> o9.
DEBUG:onnx2keras:Output 10 -> o10.
DEBUG:onnx2keras:Gathering weights to dictionary.
DEBUG:onnx2keras:Found weight classifier.bias with shape (27,).
DEBUG:onnx2

DEBUG:onnx2keras:Found weight features.14.conv.1.bias with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.14.conv.1.running_mean with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.1.running_var with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.1.weight with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.3.weight with shape (576, 1, 3, 3).
DEBUG:onnx2keras:Found weight features.14.conv.4.bias with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.4.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.14.conv.4.running_mean with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.4.running_var with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.4.weight with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.6.weight with shape (160, 576, 1, 1).
DEBUG:onnx2keras:Found weight features.14.conv

DEBUG:onnx2keras:Found weight features.3.conv.4.running_mean with shape (144,).
DEBUG:onnx2keras:Found weight features.3.conv.4.running_var with shape (144,).
DEBUG:onnx2keras:Found weight features.3.conv.4.weight with shape (144,).
DEBUG:onnx2keras:Found weight features.3.conv.6.weight with shape (24, 144, 1, 1).
DEBUG:onnx2keras:Found weight features.3.conv.7.bias with shape (24,).
DEBUG:onnx2keras:Found weight features.3.conv.7.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.3.conv.7.running_mean with shape (24,).
DEBUG:onnx2keras:Found weight features.3.conv.7.running_var with shape (24,).
DEBUG:onnx2keras:Found weight features.3.conv.7.weight with shape (24,).
DEBUG:onnx2keras:Found weight features.4.conv.0.weight with shape (144, 24, 1, 1).
DEBUG:onnx2keras:Found weight features.4.conv.1.bias with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.4.conv.1.running_mean w

DEBUG:onnx2keras:Found weight features.9.conv.3.weight with shape (384, 1, 3, 3).
DEBUG:onnx2keras:Found weight features.9.conv.4.bias with shape (384,).
DEBUG:onnx2keras:Found weight features.9.conv.4.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.9.conv.4.running_mean with shape (384,).
DEBUG:onnx2keras:Found weight features.9.conv.4.running_var with shape (384,).
DEBUG:onnx2keras:Found weight features.9.conv.4.weight with shape (384,).
DEBUG:onnx2keras:Found weight features.9.conv.6.weight with shape (64, 384, 1, 1).
DEBUG:onnx2keras:Found weight features.9.conv.7.bias with shape (64,).
DEBUG:onnx2keras:Found weight features.9.conv.7.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.9.conv.7.running_mean with shape (64,).
DEBUG:onnx2keras:Found weight features.9.conv.7.running_var with shape (64,).
DEBUG:onnx2keras:Found weight features.9.conv.7.weight with shape (64,).
DEBUG:onnx2keras:Found input i0 with shape [3, 224, 224]
DEBUG:

DEBUG:onnx2keras:node_name: 332
DEBUG:onnx2keras:node_params: {'epsilon': 9.999999747378752e-06, 'momentum': 0.8999999761581421, 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 331).
DEBUG:onnx2keras:Check input 1 (name features.1.conv.4.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.1.conv.4.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.1.conv.4.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 4 (name features.1.conv.4.running_var).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:on

DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name features.2.conv.7.bias).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 3 (name features.2.conv.7.running_mean).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 4 (name features.2.conv.7.running_var).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:Output TF Layer -> Tensor("340/Identity:0", shape=(None, 24, 56, 56), dtype=float32)
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Shape
DEBUG:onnx2keras:node_name: 341
DEBUG:onnx2keras:node_params: {'change_ordering': False, 'name_policy': 'short

DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:Output TF Layer -> 9223372036854775807
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Unsqueeze
DEBUG:onnx2keras:node_name: 357
DEBUG:onnx2keras:node_params: {'axes': [0], 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 354).
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:unsqueeze:Work with numpy types.
DEBUG:onnx2keras:Output TF Layer -> [3.]
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Unsqueeze
DEBUG:onnx2keras:node_name: 358
DEBUG:onnx2keras:node_params: {'axes': [0], 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 356).
DEBUG:onnx2keras:... found all, continue
DEBU

starts, ends [0] [3.]
before: s, e [0, 0] [0, 3.0]
after: [int(_) for _ in s], [int(_) for _ in e] [0, 0] [0, 3]
starts, ends [3.] [9223372036854775807]
before: s, e [0, 3.0] [0, 9223372036854775807]
after: [int(_) for _ in s], [int(_) for _ in e] [0, 3] [0, 9223372036854775807]


ValueError: Layer weight shape (1, 1, 23, 144) not compatible with provided weight shape (1, 1, 24, 144)

In [2]:
import onnx2keras
import torch
import onnx

torch_inputs = [torch.rand(1, 3, 224, 224, dtype=torch.float32),
                torch.zeros([1, 3, 56, 56], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32)]

onnx_path = './models/jestnet_simple.onnx'
onnx_model = onnx.load(onnx_path)
k_model = onnx2keras.onnx_to_keras(
    onnx_model, 
    input_names=[f'i{i}' for i in range(len(torch_inputs))],
    name_policy='short'
)

INFO:onnx2keras:Converter is called.
DEBUG:onnx2keras:List input shapes:
DEBUG:onnx2keras:None
DEBUG:onnx2keras:List inputs:
DEBUG:onnx2keras:Input 0 -> i0.
DEBUG:onnx2keras:Input 1 -> i1.
DEBUG:onnx2keras:Input 2 -> i2.
DEBUG:onnx2keras:Input 3 -> i3.
DEBUG:onnx2keras:Input 4 -> i4.
DEBUG:onnx2keras:Input 5 -> i5.
DEBUG:onnx2keras:Input 6 -> i6.
DEBUG:onnx2keras:Input 7 -> i7.
DEBUG:onnx2keras:Input 8 -> i8.
DEBUG:onnx2keras:Input 9 -> i9.
DEBUG:onnx2keras:Input 10 -> i10.
DEBUG:onnx2keras:List outputs:
DEBUG:onnx2keras:Output 0 -> o0.
DEBUG:onnx2keras:Output 1 -> o1.
DEBUG:onnx2keras:Output 2 -> o2.
DEBUG:onnx2keras:Output 3 -> o3.
DEBUG:onnx2keras:Output 4 -> o4.
DEBUG:onnx2keras:Output 5 -> o5.
DEBUG:onnx2keras:Output 6 -> o6.
DEBUG:onnx2keras:Output 7 -> o7.
DEBUG:onnx2keras:Output 8 -> o8.
DEBUG:onnx2keras:Output 9 -> o9.
DEBUG:onnx2keras:Output 10 -> o10.
DEBUG:onnx2keras:Gathering weights to dictionary.
DEBUG:onnx2keras:Found weight classifier.bias with shape (27,).
DEBUG:onnx2

DEBUG:onnx2keras:Found weight 358 with shape (1,).
DEBUG:onnx2keras:Found weight 359 with shape (1,).
DEBUG:onnx2keras:Found weight 387 with shape (1,).
DEBUG:onnx2keras:Found weight 388 with shape (1,).
DEBUG:onnx2keras:Found weight 389 with shape (1,).
DEBUG:onnx2keras:Found weight 396 with shape (1,).
DEBUG:onnx2keras:Found weight 397 with shape (1,).
DEBUG:onnx2keras:Found weight 398 with shape (1,).
DEBUG:onnx2keras:Found weight 418 with shape (1,).
DEBUG:onnx2keras:Found weight 419 with shape (1,).
DEBUG:onnx2keras:Found weight 420 with shape (1,).
DEBUG:onnx2keras:Found weight 427 with shape (1,).
DEBUG:onnx2keras:Found weight 428 with shape (1,).
DEBUG:onnx2keras:Found weight 429 with shape (1,).
DEBUG:onnx2keras:Found weight 457 with shape (1,).
DEBUG:onnx2keras:Found weight 458 with shape (1,).
DEBUG:onnx2keras:Found weight 459 with shape (1,).
DEBUG:onnx2keras:Found weight 466 with shape (1,).
DEBUG:onnx2keras:Found weight 467 with shape (1,).
DEBUG:onnx2keras:Found weight 4

DEBUG:onnx2keras:Check input 0 (name 331).
DEBUG:onnx2keras:Check input 1 (name 800).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name 802).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:conv:Conv with bias
DEBUG:onnx2keras:conv:2D convolution
DEBUG:onnx2keras:Output TF Layer -> Tensor("333_1/Identity:0", shape=(None, 96, 112, 112), dtype=float32)
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Clip
DEBUG:onnx2keras:node_name: 335
DEBUG:onnx2keras:node_params: {'max': 6.0, 'min': 0.0, 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 333).
DEBUG:onnx2keras:... found all, con

starts, ends [0] [3]
before: s, e [0, 0] [0, 3]
after: [int(_) for _ in s], [int(_) for _ in e] [0, 0] [0, 3]
starts, ends [3] [9223372036854775807]
before: s, e [0, 3] [0, 9223372036854775807]
after: [int(_) for _ in s], [int(_) for _ in e] [0, 3] [0, 9223372036854775807]


ValueError: Layer weight shape (1, 1, 23, 144) not compatible with provided weight shape (1, 1, 24, 144)

* **mmdnn pytorch2keras**: [does not support](https://github.com/microsoft/MMdnn/issues/794) multiple inputs, BUT YOU CAN CONCAT ALL INPUTS IN ONE TENSOR OF SHAPE  
`(1, sum(channels), 224, 224)` and then do slices to exctract the buffer inside of the model;
* mmdnn works [only with pytorch 0.4.0](https://github.com/microsoft/MMdnn/issues/426), I had a problem loading jester weights for pth0.4.0 so I stopped pushing this direction. 

**Looking towards pure keras tsm implementation.**

Can help: 
* [tf.js webcam demo](https://github.com/tensorflow/tfjs-examples/tree/master/webcam-transfer-learning)
* [tf.js native mobilenet](https://github.com/tensorflow/tfjs-models/tree/master/mobilenet)
* [tf.js-converter mobilenet demo](https://github.com/tensorflow/tfjs/tree/master/tfjs-converter/demo/mobilenet) (converted from TF)
* [onnx-tracing-vs-scripting](https://pytorch.org/docs/master/onnx.html#tracing-vs-scripting)
* [keras-js](https://github.com/transcranial/keras-js)
* https://kevin970401.github.io/etc/2019/08/21/converting-model-pth-keras.html
* https://github.com/tensorflow/tfjs/issues/2348
* https://www.npmjs.com/package/@tensorflow-models/handpose
* https://github.com/tensorflow/tfjs-models/tree/master/handpose

---

### Run model in camera loop:

* PyTorch loop:

In [46]:
# WINDOW_NAME = 'Video Gesture Recognition'

# print("Open camera...")
# cap = cv2.VideoCapture(0)

# print(cap)

# # set a lower resolution for speed up
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# # env variables
# full_screen = False
# cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
# cv2.resizeWindow(WINDOW_NAME, 640, 480)
# cv2.moveWindow(WINDOW_NAME, 0, 0)
# cv2.setWindowTitle(WINDOW_NAME, WINDOW_NAME)


# t = None
# index = 0
# buffer = (
#     torch.zeros([1, 3, 56, 56]),
#     torch.zeros([1, 4, 28, 28]),
#     torch.zeros([1, 4, 28, 28]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 12, 14, 14]),
#     torch.zeros([1, 12, 14, 14]),
#     torch.zeros([1, 20, 7, 7]),
#     torch.zeros([1, 20, 7, 7])
# )

# idx = 0
# history = [2, 2]
# history_logit = []
# history_timing = []

# i_frame = -1

# print("Ready!")
# while True:
#     i_frame += 1
#     _, img = cap.read()  # (480, 640, 3) 0 ~ 255
#     if i_frame % 2 == 0:  # skip every other frame to obtain a suitable frame rate
#         t1 = time.time()
#         img_tran = torch.from_numpy(transform(img)).float().contiguous()
#         with torch.no_grad():
#             outputs = torch_module(img_tran, *buffer)  # was input_var earlier
#             feat, buffer = outputs[0], outputs[1:]
#         if SOFTMAX_THRES > 0:
#             feat_np = feat.numpy().reshape(-1)
#             feat_np -= feat_np.max()
#             softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
#             print(max(softmax))
#             if max(softmax) > SOFTMAX_THRES:
#                 idx_ = np.argmax(feat.numpy(), axis=1)[0]
#             else:
#                 idx_ = idx
#         else:
#             idx_ = np.argmax(feat.numpy(), axis=1)[0]
#         if HISTORY_LOGIT:
#             history_logit.append(feat.numpy())
#             history_logit = history_logit[-12:]
#             avg_logit = sum(history_logit)
#             idx_ = np.argmax(avg_logit, axis=1)[0]
#         idx, history = process_output(idx_, history)
#         t2 = time.time()
#         print(f"{index} {catigories[idx]}")
#         current_time = t2 - t1
#     img = cv2.resize(img, (640, 480))
#     img = img[:, ::-1]
#     height, width, _ = img.shape
#     label = np.zeros([height // 10, width, 3]).astype('uint8') + 255
#     cv2.putText(label, 'Prediction: ' + catigories[idx],
#                 (0, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     cv2.putText(label, '{:.1f} Vid/s'.format(1 / current_time),
#                 (width - 170, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     img = np.concatenate((img, label), axis=0)
#     cv2.imshow(WINDOW_NAME, img)
#     key = cv2.waitKey(1)
#     if key & 0xFF == ord('q') or key == 27:  # exit
#         break
#     elif key == ord('F') or key == ord('f'):  # full screen
#         print('Changing full screen option!')
#         full_screen = not full_screen
#         if full_screen:
#             print('Setting FS!!!')
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_FULLSCREEN)
#         else:
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_NORMAL)
#     if t is None:
#         t = time.time()
#     else:
#         nt = time.time()
#         index += 1
#         t = nt

In [47]:
# cap.release()
# cv2.destroyAllWindows()

* ONNX Runtime loop:

In [19]:
import onnxruntime
ort_session = onnxruntime.InferenceSession(ONNX_SIMPLE_MODEL_PATH)
input_names = [ort_session.get_inputs()[i].name for i in range(len(ort_session.get_inputs()))]
output_names = [ort_session.get_outputs()[i].name for i in range(len(ort_session.get_outputs()))]

WINDOW_NAME = 'Video Gesture Recognition'

print("Open camera...")
cap = cv2.VideoCapture(0)

print(cap)

# set a lower resolution for speed up
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# env variables
full_screen = False
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 640, 480)
cv2.moveWindow(WINDOW_NAME, 0, 0)
cv2.setWindowTitle(WINDOW_NAME, WINDOW_NAME)


t = None
index = 0
np_buffer = [
    np.zeros([1, 3, 56, 56]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 20, 7, 7]),
    np.zeros([1, 20, 7, 7])
]
np_buffer = {f'i{i+1}': x.astype(np.float32) for i, x in enumerate(np_buffer)}

idx = 0
history = [2, 2]
history_logit = []
history_timing = []

i_frame = -1

print("Ready!")
while True:
    i_frame += 1
    _, img = cap.read()  # (480, 640, 3) 0 ~ 255
    if i_frame % 2 == 0:  # skip every other frame to obtain a suitable frame rate
        t1 = time.time()
        img_tran = transform(img)
        with torch.no_grad():
            np_buffer.update({'i0': img_tran})
            np_buffer = {k: v.astype(np.float32) for k, v in np_buffer.items()}
            outputs = ort_session.run(output_names, np_buffer)
            feat, np_buffer_values = outputs[0], outputs[1:]
            np_buffer = {f'i{i+1}': np_buffer_values[i] for i in range(len(np_buffer_values))}
        if SOFTMAX_THRES > 0:
            feat_np = feat.reshape(-1)
            feat_np -= feat_np.max()
            softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
            print(max(softmax))
            if max(softmax) > SOFTMAX_THRES:
                idx_ = np.argmax(feat, axis=1)[0]
            else:
                idx_ = idx
        else:
            idx_ = np.argmax(feat, axis=1)[0]
        if HISTORY_LOGIT:
            history_logit.append(feat)
            history_logit = history_logit[-12:]
            avg_logit = sum(history_logit)
            idx_ = np.argmax(avg_logit, axis=1)[0]
        idx, history = process_output(idx_, history)
        t2 = time.time()
        print(f"{index} {catigories[idx]}")
        current_time = t2 - t1
    img = cv2.resize(img, (640, 480))
    img = img[:, ::-1]
    height, width, _ = img.shape
    label = np.zeros([height // 10, width, 3]).astype('uint8') + 255
    cv2.putText(label, 'Prediction: ' + catigories[idx],
                (0, int(height / 16)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 0, 0), 2)
    cv2.putText(label, '{:.1f} Vid/s'.format(1 / current_time),
                (width - 170, int(height / 16)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 0, 0), 2)
    img = np.concatenate((img, label), axis=0)
    cv2.imshow(WINDOW_NAME, img)
    key = cv2.waitKey(1)
    if key & 0xFF == ord('q') or key == 27:  # exit
        break
    elif key == ord('F') or key == ord('f'):  # full screen
        print('Changing full screen option!')
        full_screen = not full_screen
        if full_screen:
            print('Setting FS!!!')
            cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
                                  cv2.WINDOW_FULLSCREEN)
        else:
            cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
                                  cv2.WINDOW_NORMAL)
    if t is None:
        t = time.time()
    else:
        nt = time.time()
        index += 1
        t = nt

Open camera...
<VideoCapture 0x15d2c7e30>
Ready!
0 No gesture
1 Stop Sign
3 Stop Sign
5 Shaking Hand
7 Shaking Hand
9 Shaking Hand
11 Shaking Hand
13 Shaking Hand
15 Shaking Hand
17 Shaking Hand
19 Shaking Hand
21 Shaking Hand
23 Shaking Hand
25 Shaking Hand
27 Shaking Hand
29 Shaking Hand
31 Shaking Hand
33 Shaking Hand
35 Shaking Hand
37 Shaking Hand
39 Shaking Hand
41 Swiping Left
43 Swiping Left
45 Swiping Left
47 Swiping Left
49 Swiping Left
51 Swiping Left
53 Swiping Left
55 Swiping Left
57 Swiping Left
59 Swiping Right
61 Swiping Right
63 Swiping Right
65 Swiping Right
67 Swiping Right
69 Swiping Right
71 Swiping Right
73 Swiping Right
75 No gesture
77 No gesture
79 Swiping Down
81 Swiping Down
83 Swiping Down
85 Swiping Down
87 Swiping Up
89 Swiping Up
91 Swiping Down
93 Swiping Down
95 Swiping Up
97 Swiping Up
99 Swiping Up
101 Swiping Up
103 Swiping Up
105 Swiping Down
107 Swiping Down
109 Drumming Fingers
111 Drumming Fingers
113 Drumming Fingers
115 Drumming Fingers
117 Dru

777 Sliding Two Fingers Up
779 Sliding Two Fingers Up
781 Sliding Two Fingers Up
783 Sliding Two Fingers Up
785 Sliding Two Fingers Up
787 Sliding Two Fingers Up
789 Sliding Two Fingers Up
791 Sliding Two Fingers Up
793 Sliding Two Fingers Up
795 Sliding Two Fingers Up
797 Sliding Two Fingers Up
799 Sliding Two Fingers Down
801 Sliding Two Fingers Down
803 Zooming Out With Two Fingers
805 Zooming Out With Two Fingers
807 Zooming In With Two Fingers
809 Zooming In With Two Fingers
811 Zooming In With Two Fingers
813 Zooming In With Two Fingers
815 Zooming In With Two Fingers
817 Zooming In With Two Fingers
819 Zooming In With Two Fingers
821 Zooming In With Two Fingers
823 Zooming In With Two Fingers
825 Zooming In With Two Fingers
827 Zooming Out With Two Fingers
829 Zooming Out With Two Fingers
831 Zooming Out With Two Fingers
833 Zooming Out With Two Fingers
835 Zooming Out With Two Fingers
837 Zooming Out With Two Fingers
839 Zooming Out With Two Fingers
841 Zooming In With Two Fing

KeyboardInterrupt: 

In [58]:
cap.release()
cv2.destroyAllWindows()